# Walk-forward strategy validation

Compare selecting limits for terminal execution value versus selecting them for Calmar ratio, then test each policy on unseen years.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from retail_sp500.daily_data import daily_data_summary, load_or_fetch_twelve_data_daily

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

from retail_sp500.limit_plotting import walk_forward_calmar_figure, walk_forward_discount_figure
from retail_sp500.limit_portfolio import walk_forward_recurring_limit_selection

In [ ]:
SYMBOL = "SPY"
START_DATE = "2007-06-01"
CACHE_PATH = Path("data/processed/spy_daily_1day.csv")
REFRESH = False

daily = load_or_fetch_twelve_data_daily(
    os.getenv("TWELVE_DATA_API_KEY"),
    cache_path=CACHE_PATH,
    refresh=REFRESH,
    symbol=SYMBOL,
    start_date=START_DATE,
)

source = daily_data_summary(daily, symbol=SYMBOL)
print(source)
assert source["interval"] == "1day"
assert daily.index.max() <= pd.Timestamp.today().normalize()
daily.tail()

## Run both selection policies

In [ ]:
DISCOUNTS = np.arange(0.0, 0.0301, 0.001)
TRAIN_YEARS = 5
TEST_YEARS = 1
MAX_WAIT_SESSIONS = 5

walk_forward = pd.concat(
    [
        walk_forward_recurring_limit_selection(
            daily,
            DISCOUNTS,
            train_years=TRAIN_YEARS,
            test_years=TEST_YEARS,
            step_years=1,
            max_wait_sessions=MAX_WAIT_SESSIONS,
            monthly_contribution=1_000,
            selection_metric=metric,
        )
        for metric in ["ending_excess_value", "calmar_ratio"]
    ],
    ignore_index=True,
)

walk_forward

In [ ]:
walk_forward_discount_figure(walk_forward).show()

In [ ]:
walk_forward_calmar_figure(walk_forward).show()

In [ ]:
px.bar(
    walk_forward,
    x="test_start",
    y="test_ending_excess_value",
    color="selection_metric",
    barmode="group",
    title="Out-of-sample terminal execution value by selection policy",
    labels={
        "test_start": "Test year",
        "test_ending_excess_value": "Excess ending value",
        "selection_metric": "Training objective",
    },
).show()

## Out-of-sample summary

In [ ]:
walk_forward.groupby("selection_metric").agg(
    folds=("test_start", "size"),
    median_selected_discount=("selected_discount", "median"),
    mean_test_calmar=("test_calmar_ratio", "mean"),
    median_test_calmar=("test_calmar_ratio", "median"),
    positive_excess_fold_rate=("test_ending_excess_value", lambda values: (values > 0).mean()),
    calmar_beats_baseline_rate=("test_calmar_ratio", lambda values: (
        values.to_numpy() > walk_forward.loc[values.index, "test_baseline_calmar_ratio"].to_numpy()
    ).mean()),
)

## Decision rule

Prefer broad parameter regions and policies that remain competitive across unseen folds. A single full-sample Calmar maximum is not sufficient evidence.